# FP&A Variance Copilot — Analytical Methodology

> **Who this is for:** Engineers, data scientists, and FP&A practitioners who want to understand not just *what* this tool does, but *why* each design decision was made — and the trade-offs considered.
>
> Run each cell in order. All outputs should reproduce exactly from the included sample data.

---

## 1. Introduction — The Problem With Variance Reporting

Every finance team produces a budget vs. actuals report at the end of a period. Almost none of them are useful.

The typical FP&A variance report is a spreadsheet with hundreds of rows, a Variance ($) column, and maybe a Variance (%) column. The analyst spends a day copy-pasting numbers into a PowerPoint deck. The CFO spends 20 minutes reading it, skips to the bottom line, and asks the same three questions they ask every quarter.

**The core problems:**

| Problem | Why it's painful |
|---------|------------------|
| **Signal-to-noise ratio** | A 20-row P&L shows 20 variances. Maybe 4 matter. The other 16 create cognitive load with no payoff. |
| **Sign convention ambiguity** | A negative variance on a cost line is *good*. A negative variance on a revenue line is *bad*. Spreadsheets show you the number, not the judgment. |
| **Missing root causes** | The report says "Cloud Infrastructure: +$132K over budget." It doesn't say why, or what to do about it. |
| **No forward signal** | Historical variance reports describe the past. CFOs need to know if this quarter's miss is a one-off or a structural problem that will recur. |
| **Tone-deaf presentation** | The same numbers require different framing for a board (strategic implications), a team (action items), and investors (beat/miss narrative). A single spreadsheet serves none of these audiences well. |

**What this tool builds instead:**

1. **Materiality filtering** — surfaces only the variances that exceed significance thresholds (configurable)
2. **Favorability scoring** — applies the correct sign convention per category, so red/green means something
3. **Severity classification** — four-band system (Critical/Major/Moderate/Minor) for at-a-glance triage
4. **Waterfall bridge** — shows the budget-to-actual path as a visual story, not a column of numbers
5. **Tone-controlled commentary** — three distinct AI outputs from the same underlying data
6. **Forward projection** — run-rate modeling to flag structural risks before they become next quarter's problem

Each of these is implemented as a pure-Python module with no UI coupling, so the logic is independently testable and reusable.

---

## 2. Data Structure

### 2.1 Setup and imports

In [ ]:
import sys
import os

# Make sure the project root is on the path so we can import the modules directly
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('Project root:', project_root)

### 2.2 Load the sample data

In [ ]:
csv_path = os.path.join(project_root, 'sample_data', 'actuals_vs_budget_q4_2024.csv')
df_raw = pd.read_csv(csv_path)
print(f'Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
df_raw

### 2.3 Schema

The tool requires exactly four columns (case-insensitive):

| Column | Type | Description |
|--------|------|-------------|
| `Line Item` | string | The P&L line description — e.g., "Cloud Infrastructure & Hosting" |
| `Category` | string | The grouping — drives the sign convention (see §2.4) |
| `Budget` | float | The planned dollar amount for the period |
| `Actual` | float | The realised dollar amount for the period |

`Period` is optional. If present, it's informational only — the reporting period label comes from the UI sidebar, not this column.

**Why these four, and not more?** The design goal was the lowest possible barrier to upload. Every FP&A team has a spreadsheet that contains at minimum Line Item, Category, Budget, and Actual. Requiring a Period column, or a Department column, or a GL code would exclude ~30% of real-world uploads. Everything this tool needs to compute — variances, severity, waterfall positions, projections — can be derived from these four.

In [ ]:
print('Data types:')
print(df_raw.dtypes)
print()
print('Category values in this dataset:')
print(df_raw['Category'].value_counts().to_string())

### 2.4 Sign conventions — the key conceptual decision

This is the most important design decision in the entire tool, and the one most spreadsheets get wrong by omission.

**The problem:** `Variance ($) = Actual − Budget` produces a number. That number means something *different* depending on whether the line item is revenue or a cost:

- **Revenue line:** Actual $4.65M vs. Budget $4.2M → Variance +$450K → **Good.** More revenue is better.
- **Cost line:** Actual $784K vs. Budget $620K → Variance +$164K → **Bad.** Higher spend is worse.

The naive approach (colour everything positive green and negative red) fails for cost lines — you'd colour an overspend *green*.

**Our approach:** The `Category` string determines the sign convention. Any category containing a revenue keyword gets the revenue convention; everything else gets the cost convention.

In [ ]:
from config import REVENUE_CATEGORY_KEYWORDS
print('Revenue keywords (any category containing one of these uses revenue convention):')
for kw in REVENUE_CATEGORY_KEYWORDS:
    print(f'  "{kw}"')

In [ ]:
# Show how each category in our sample resolves
categories = df_raw['Category'].unique()

print(f'{"Category":<30} {"Convention":<12} {"Matching keyword"}')
print('-' * 60)
for cat in sorted(categories):
    cat_lower = cat.lower()
    matching = [kw for kw in REVENUE_CATEGORY_KEYWORDS if kw in cat_lower]
    if matching:
        print(f'{cat:<30} {"REVENUE":<12} "{matching[0]}"')
    else:
        print(f'{cat:<30} {"COST":<12} (default)')

**Why keyword matching on the category string, not an explicit flag column?**

Two reasons:
1. It keeps the upload schema at four columns — no "sign_convention" column for users to fill in and get wrong.
2. Category names in real-world P&L exports are generally self-describing. A category called "Revenue" or "Other Income" will match the keyword list reliably.

The keywords are configurable in `config.py`. If a user has a non-standard category name (e.g., "TopLine" for revenue), they can add it to `REVENUE_CATEGORY_KEYWORDS` without changing any other code.

---

## 3. Variance Computation

The computation pipeline in `variance_engine.compute_variances()` adds five columns to the raw DataFrame. Let's walk through each one.

### 3.1 Run the full pipeline

In [ ]:
from variance_engine import validate_csv, compute_variances, summarize_by_category

# Validate first (this also canonicalises column names)
ok, msg = validate_csv(df_raw.copy())
print(f'Validation: {"PASS" if ok else "FAIL"} — {msg if not ok else "all checks passed"}')

# Compute variances
enriched = compute_variances(df_raw.copy())
print(f'\nNew columns added: {[c for c in enriched.columns if c not in df_raw.columns]}')
enriched[['Line Item', 'Category', 'Budget', 'Actual', 'Variance ($)', 'Variance (%)', 'Favorable', 'Severity', 'Material']]

### 3.2 Step-by-step: each column

#### Variance ($)

In [ ]:
# Variance ($) = Actual - Budget
# Always arithmetic. The sign is NOT adjusted for convention — that's Favorable's job.
demo = df_raw[['Line Item', 'Category', 'Budget', 'Actual']].copy()
demo['Variance ($) [manual]'] = demo['Actual'] - demo['Budget']
demo['Matches engine'] = demo['Variance ($) [manual]'] == enriched['Variance ($)']

print('Spot check — manual vs. engine computation:')
demo[['Line Item', 'Variance ($) [manual]', 'Matches engine']].head(6)

#### Variance (%)

A subtle but important choice: the denominator is `|Budget|` (absolute value), not `Budget`.

Why? Some line items legitimately have a negative budget (e.g., a contra-revenue account). Using the signed budget would flip the sign of the percentage for those rows, making the output misleading. The absolute value denominator ensures the percentage always describes magnitude, and the `Favorable` flag handles directional interpretation separately.

In [ ]:
# Variance (%) = Variance ($) / |Budget| × 100
# NaN when Budget == 0 (avoids division by zero; these rows are treated as Minor severity)

demo2 = enriched[['Line Item', 'Budget', 'Variance ($)', 'Variance (%)']].copy()
demo2['Manual %'] = np.where(
    demo2['Budget'] != 0,
    demo2['Variance ($)'] / demo2['Budget'].abs() * 100,
    np.nan
)
demo2['Delta'] = (demo2['Manual %'] - demo2['Variance (%)']).abs()
print(f'Max floating-point delta: {demo2["Delta"].max():.2e}')  # should be ~0
demo2[['Line Item', 'Budget', 'Variance ($)', 'Manual %', 'Variance (%)']].round(2)

#### Favorable (sign-adjusted)

This column is the payoff of the sign convention work in §2.4. It translates arithmetic variance into operational judgment.

In [ ]:
# Show the logic explicitly for a few rows
check = enriched[['Line Item', 'Category', 'Variance ($)', 'Favorable']].copy()

from config import REVENUE_CATEGORY_KEYWORDS
check['Convention'] = check['Category'].apply(
    lambda c: 'REVENUE' if any(k in c.lower() for k in REVENUE_CATEGORY_KEYWORDS) else 'COST'
)
check['Expected Favorable'] = check.apply(
    lambda r: (r['Variance ($)'] >= 0) if r['Convention'] == 'REVENUE'
              else (r['Variance ($)'] <= 0),
    axis=1
)
check['Correct'] = check['Favorable'] == check['Expected Favorable']

# Show the interesting cases — where convention matters
print('Cases where REVENUE convention changes the answer vs. naive positive=good:')
interesting = check[
    (check['Convention'] == 'REVENUE') & (check['Variance ($)'] < 0) |
    (check['Convention'] == 'COST') & (check['Variance ($)'] > 0)
]
print(interesting[['Line Item', 'Category', 'Variance ($)', 'Convention', 'Favorable']].to_string(index=False))

#### Severity Classification

Severity maps the absolute variance percentage to a four-band label. The bands are designed to align with how a CFO naturally segments variances:

| Band | Threshold | Interpretation |
|------|-----------|----------------|
| **Critical** | \|Var%\| ≥ 15% | Requires CFO attention this week — investigate root cause |
| **Major** | \|Var%\| ≥ 10% | Needs explanation in the board pack — flagged for follow-up |
| **Moderate** | \|Var%\| ≥ 5% | Worth noting — context-dependent whether action is needed |
| **Minor** | \|Var%\| < 5% | Within normal operating variance — no action required |

Note: Severity is based on *percentage* only, not absolute dollars. A 20% miss on a $10K line is Critical severity, but the $2K dollar impact is immaterial. That's why we also have a separate Materiality flag — severity and materiality are orthogonal signals.

In [ ]:
from config import SEVERITY_BANDS

print('Severity band definitions:')
print(f'{"Band":<12} {"Lower bound (|Var%|)"}')
print('-' * 35)
for label, threshold in SEVERITY_BANDS:
    print(f'{label:<12} >= {threshold*100:.0f}%')

print()
print('Distribution in Q4 2024 sample:')
print(enriched['Severity'].value_counts().reindex(['Critical', 'Major', 'Moderate', 'Minor']).to_string())

In [ ]:
# Verify the band assignment logic for a few rows
sev_check = enriched[['Line Item', 'Variance (%)', 'Severity']].copy()
sev_check['|Var%|'] = sev_check['Variance (%)'].abs()

def expected_severity(abs_pct):
    if pd.isna(abs_pct):
        return 'Minor'
    for label, threshold in SEVERITY_BANDS:
        if abs_pct / 100 >= threshold:
            return label
    return 'Minor'

sev_check['Expected'] = sev_check['|Var%|'].apply(expected_severity)
sev_check['Match'] = sev_check['Severity'] == sev_check['Expected']
print(f'All severity assignments match: {sev_check["Match"].all()}')
sev_check[['Line Item', '|Var%|', 'Severity', 'Expected']].sort_values('|Var%|', ascending=False)

#### Materiality — Why Two Thresholds?

A variance is **material** if it breaches *either* threshold:
- `|Var%| >= MATERIALITY_PCT` (default: 5%), **OR**
- `|Var$| >= MATERIALITY_ABS` (default: $50,000)

**The case for a dual threshold:**

A percentage-only threshold has a pathological failure on large budget lines. A $5M budget line with a $200K miss is only 4% — below the materiality threshold — but $200K is a meaningful dollar amount that a CFO should see. The absolute dollar floor catches these.

Conversely, a dollar-only threshold is noisy on small budget lines. A $10K line with a $6K miss is 60% deviation — structurally something is wrong with the budget — but a $6K dollar impact is trivially immaterial. The percentage threshold catches this.

The OR logic means a variance needs to clear *either* bar to make the material items list — the analyst sees it if the miss is large in dollars OR large as a percentage of plan.

In [ ]:
from config import MATERIALITY_PCT, MATERIALITY_ABS

print(f'Default thresholds:')
print(f'  % threshold : |Var%| >= {MATERIALITY_PCT*100:.0f}%')
print(f'  $ threshold : |Var$| >= ${MATERIALITY_ABS:,}')
print()

mat = enriched[['Line Item', 'Category', 'Variance ($)', 'Variance (%)', 'Material']].copy()
mat['|Var%|']  = mat['Variance (%)'].abs()
mat['|Var$|']  = mat['Variance ($)'].abs()
mat['By %']    = mat['|Var%|'] >= MATERIALITY_PCT * 100
mat['By $']    = mat['|Var$|'] >= MATERIALITY_ABS
mat['Either']  = mat['By %'] | mat['By $']

# Cases where the dual threshold matters — one fires but not the other
edge_cases = mat[mat['By %'] != mat['By $']]
print(f'Edge cases where % and $ thresholds disagree ({len(edge_cases)} rows):')
print(edge_cases[['Line Item', '|Var%|', '|Var$|', 'By %', 'By $', 'Material']].to_string(index=False))

In [ ]:
# Category-level summary — what does compute_variances surface?
cat_df = summarize_by_category(enriched)
print(f'Category summary ({len(cat_df)} categories, sorted by |Var$|):')
cat_df[['Category', 'Budget ($)', 'Actual ($)', 'Variance ($)', 'Variance (%)', 'Material Items', 'Dominant Severity']].round(1)

**Why recalculate Variance (%) at the category level rather than averaging row-level percentages?**

Averaging percentages is a classic mistake. If line A has a 50% variance on a $100K budget and line B has a 2% variance on a $5M budget, the simple average (26%) wildly misrepresents the category. We compute category-level percentage as `ΣVariance($) / |ΣBudget($)| × 100` — it weights by budget size, which is what a CFO would do mentally.

---

## 4. Waterfall Construction

The waterfall (bridge) chart is the single most informative visualisation in variance analysis. It shows the narrative path from the budgeted total to the actual total through each contributing variance — a CFO can read the story in seconds.

### 4.1 Why a waterfall chart beats a variance table

A table of variances requires the reader to mentally rank and aggregate. A waterfall chart does that work for them:
- Bars to the right = favorable (budget was surpassed or costs were below plan)
- Bars to the left = unfavorable (budget was missed or costs overran)
- Bar height = relative impact — the eye immediately identifies which variance matters most
- The final bar confirms the arithmetic closes to Actual

### 4.2 Bridge data construction

In [ ]:
from variance_engine import build_waterfall_data
from config import WATERFALL_MIN_IMPACT_PCT

waterfall_df = build_waterfall_data(enriched)

print(f'Min impact threshold: {WATERFALL_MIN_IMPACT_PCT*100:.1f}% of total budget')
total_budget = enriched['Budget'].sum()
print(f'Total budget: ${total_budget:,.0f}')
print(f'Dollar cutoff for bridge bars: ${total_budget * WATERFALL_MIN_IMPACT_PCT:,.0f}')
print()
print('Bridge data:')
waterfall_df

### 4.3 Structural design of the bridge

The bridge has three types of bars:

1. **`absolute` bars** — Budget (opening) and Actual (closing). These set the anchors of the chart.
2. **`relative` bars** — Individual variance bridges. Each bar is the delta, not a running total. Plotly's `go.Waterfall` handles the stacking automatically.
3. **`Other (net)` bar** — A residual bar that absorbs the variance from all sub-threshold line items. This is crucial: without it, the bridge would not arithmetically close to Actual.

**The residual bar problem:** If we included only the top-N variances, their sum might not equal `Actual − Budget`. The residual catches the difference. The condition `abs(residual) > 1` ignores floating-point dust.

In [ ]:
# Verify the bridge closes
total_actual = enriched['Actual'].sum()
relative_bars = waterfall_df[waterfall_df['measure'] == 'relative']['value'].sum()
bridge_actual = total_budget + relative_bars

print(f'Total Budget          : ${total_budget:>15,.0f}')
print(f'Sum of bridge bars    : ${relative_bars:>+15,.0f}')
print(f'Budget + bridge       : ${bridge_actual:>15,.0f}')
print(f'Total Actual          : ${total_actual:>15,.0f}')
print(f'Difference (rounding) : ${bridge_actual - total_actual:>+15,.2f}')
print(f'\nBridge closes: {abs(bridge_actual - total_actual) < 1}')

### 4.4 Render the waterfall chart

In [ ]:
from charts import plot_waterfall

fig = plot_waterfall(waterfall_df, title='Q4 2024 — Budget-to-Actual Bridge')
fig.show()

### 4.5 Chart design decisions

- **Color coding:** Green (`#16A34A`) for favorable moves, red (`#DC2626`) for unfavorable, teal (`#0D9488`) for the Budget and Actual totals. These map directly to the `Favorable` flag in the data — there's no separate coloring logic.

- **Label rotation:** When there are more than 8 bars, x-axis labels are angled at 35°. Below 8 bars, they stay horizontal. This prevents overlap on dense charts without requiring manual intervention.

- **Compact currency labels:** Bridge bar labels show `+$1.2M` style notation (compact). The hover tooltip shows the full precision. This keeps the chart readable at presentation size.

- **Font:** Calibri throughout — the standard for financial presentations in most corporate environments. Falls back to Segoe UI, then Arial.

---

## 5. Commentary Generation

The commentary module does something conceptually unusual: the same underlying data produces three structurally and tonally distinct outputs, each optimised for a different reader. This section explains the prompt engineering decisions.

### 5.1 The three tones — why they're genuinely different

Most tools that offer "tone" selection just adjust adjective strength. Our three tones have *structurally different* outputs:

| Aspect | Board Presentation | Internal Team Update | Investor Call Prep |
|--------|-------------------|---------------------|--------------------|
| **Length** | Medium — strategic depth | Short (<300 words) | Longest — includes Q&A prep |
| **Structure** | 5 sections inc. Recommendations | 5 sections inc. To-Dos | 6 sections inc. Language Guidance |
| **Opening** | Bottom Line (net $, judgment) | TL;DR (one bold sentence) | Key Messages (3 investor bullets) |
| **Unique section** | Investment Areas | Owner assignments | Anticipated Q&A |
| **Hedging level** | Low — state causes directly | None — go straight to numbers | High — IR-standard hedging required |
| **Audience assumption** | Strategy-level; numbers as context | Knows the business jargon | Will hold you to every word |

### 5.2 The system prompt — show the actual text

In [ ]:
from commentary import _SYSTEM_PROMPTS, TONES

for tone in TONES:
    print(f'{'='*70}')
    print(f'TONE: {tone}')
    print('='*70)
    print(_SYSTEM_PROMPTS[tone])
    print()

### 5.3 Prompt engineering decisions

#### Decision 1: Structured sections, not free-form

Each system prompt specifies an *exact* section structure with numbered sections and named headers. This is deliberate:

- **Without structure:** The model produces a well-written paragraph that might lead with revenue, might lead with costs, might lead with an overview — unpredictable for repeated use.
- **With structure:** Every Board output has BOTTOM LINE first. Every Team output has TL;DR first. Users build a mental model of where to look.

The tradeoff is that structured prompts occasionally produce slightly awkward transitions when the data is sparse in one section. We judged consistency more valuable than occasional awkwardness.

#### Decision 2: Theme-based grouping beats line-by-line enumeration

A naive commentary prompt would ask: "Describe each material variance." This produces output like:
> *Product License Revenue was -$490K (-12.9%). Cloud Infrastructure was +$132K (+27.5%). Digital Marketing was +$164K (+26.5%).*

That's a ledger recitation, not analysis. Our prompts instead ask for theme-based grouping:
- Board: "Group all revenue variances" → one coherent revenue narrative
- Board: "Split costs into structural overruns vs. one-time spend" → the CFO-relevant distinction
- Investor: "Beat/miss framing vs. budget-as-guidance" → the IR-standard frame

Grouping forces the model to synthesize, not enumerate.

#### Decision 3: Root cause labeling

Every Board section rule includes: *"Each finding needs a root-cause hypothesis, clearly labelled as such."*

The phrase "clearly labelled as such" is load-bearing. It prevents the model from stating hypotheses as facts ("Cloud Infrastructure overran because of AWS price increases") and instead forces the epistemic qualifier ("Root cause hypothesis: AWS pricing changes in Q4..."). This matters because we're giving the model numbers, not business context — it should signal uncertainty.

#### Decision 4: The data context format

In [ ]:
from commentary import build_data_context

context = build_data_context(enriched, cat_df, 'Q4 2024')
print(context)

**Why this format?**

The data context is serialised as structured plain text rather than JSON or a Markdown table. The reasons:

1. **Token efficiency:** A markdown table for 20 rows with 6 columns uses ~60% more tokens than pipe-delimited rows. We're sending this in every call.
2. **LLM readability:** Language models trained on prose and code handle labeled fields (`Budget $4.2M | Actual $4.65M`) more reliably than JSON object parsing inside the context.
3. **Material flagging:** The `[MATERIAL]` annotation at the end of each material row is a direct signal to the model about which items deserve attention. Without it, the model must infer materiality from the numbers — less reliable.
4. **Category summary:** Included separately as an aggregate roll-up so the model can answer category-level questions without summing across rows.

### 5.4 Rule-based fallback

When no API key is provided, the tool falls back to a rule-based commentary that computes values directly from the DataFrame and assembles them into structured text. This was built for three reasons:

1. **Demo / portfolio**: The tool is fully functional without an API key — useful for showcasing in interviews.
2. **Offline use**: FP&A teams in regulated industries sometimes can't use external AI APIs. Rule-based output still surfaces the right structure and numbers.
3. **Consistency testing**: The rule-based output is deterministic, making it easy to verify correctness. The AI output can then be evaluated against it.

The rule-based commentary uses the same section structure as the AI prompts — the sections are named identically, appear in the same order, and follow the same tone conventions. A reader shouldn't be able to tell which output came from which path (within reason).

In [ ]:
from commentary import generate_commentary

# Generate rule-based commentary for each tone (no API key needed)
for tone in TONES:
    text = generate_commentary(enriched, cat_df, 'Q4 2024', tone=tone, api_key='')
    word_count = len(text.split())
    print(f'{tone}: {word_count} words')

In [ ]:
# Show the Internal Team Update output in full — it's the shortest and most concrete
team_text = generate_commentary(enriched, cat_df, 'Q4 2024', tone='Internal Team Update', api_key='')
print(team_text)

---

## 6. Forward Projection

Historical variance analysis describes what happened. The forward projection answers the CFO's real question: *"If nothing changes, what does next quarter look like?"*

### 6.1 The run-rate assumption — what it means and why

The projection methodology is deliberately simple:

- **Material variances** → Project the same dollar variance into next quarter (`Projected Variance ($) = Current Variance ($)`)
- **Non-material variances** → Assume they normalize to zero (`Projected Variance ($) = 0`)

**Why dollar run-rate instead of percentage?**

Consider Cloud Infrastructure at +$132K over budget. Two approaches:
- Dollar run-rate: "Next quarter, assume +$132K overrun on whatever the Q1 budget is."
- Percentage run-rate: "Next quarter, assume 27.5% overrun on the Q1 budget."

The dollar approach is more conservative for cost overruns (the absolute dollar impact is clearer) and more natural for revenue misses (revenue targets are typically set in dollars, not percentages). The percentage approach inflates the projected variance when next-quarter budget is higher — the dollar approach is harder to be wrong about.

The projection module's docstring explicitly notes this trade-off and where a growth-adjusted variant would be preferred.

**Why do non-material variances normalize to zero?**

Non-material variances are, by definition, noise below the significance threshold. Projecting them forward would amplify noise rather than signal. The conservative choice is to assume they mean-revert — if a Minor variance persists and becomes material next quarter, it will correctly surface as material in the next analysis cycle.

In [ ]:
from projection import project_next_quarter, summarize_projection, next_quarter_label

next_q = next_quarter_label('Q4 2024')
print(f'Current period: Q4 2024')
print(f'Projected period: {next_q}')

# Project with 0% budget growth (same budget next quarter)
proj_df = project_next_quarter(enriched, next_q_growth_pct=0.0)

# Show projection columns for material items
proj_cols = [
    'Line Item', 'Material',
    'Variance ($)',          # current quarter
    'Projected Variance ($)', # next quarter projection
    'Cumulative Variance ($)',
    'Cumulative Variance (%)',
    'At Risk',
]
proj_df[proj_cols].round(1)

In [ ]:
# Verify the run-rate logic: material items should have Projected = Current
material_rows = proj_df[proj_df['Material']]
non_material_rows = proj_df[~proj_df['Material']]

# For material items, Projected Variance ($) == Variance ($)
material_match = (material_rows['Projected Variance ($)'] == material_rows['Variance ($)']).all()

# For non-material items, Projected Variance ($) == 0
non_material_zero = (non_material_rows['Projected Variance ($)'] == 0.0).all()

print(f'Material items: Projected = Current Variance? {material_match}')
print(f'Non-material items: Projected = 0?            {non_material_zero}')
print()
print(f'Material items count:     {len(material_rows)}')
print(f'Non-material items count: {len(non_material_rows)}')
print(f'Non-material items that would normalize next quarter:')
non_material_rows[non_material_rows['Variance ($)'] != 0][['Line Item', 'Variance ($)', 'Projected Variance ($)']].to_string(index=False)

### 6.2 Budget growth adjustment

The `next_q_growth_pct` parameter scales every line item's next-quarter budget uniformly. This reflects the most common FP&A scenario: budgets are revised upward by a growth factor each quarter.

In [ ]:
# Compare projections at 0%, 5%, and 10% budget growth
scenarios = {}
for growth in [0.0, 5.0, 10.0]:
    p = project_next_quarter(enriched, next_q_growth_pct=growth)
    s = summarize_projection(p, 'Q4 2024', next_q)
    scenarios[f'{growth:.0f}% growth'] = {
        'Next Q Budget ($M)':    round(s['projected_net_budget'] / 1e6, 2),
        'Proj. Net Variance ($K)': round(s['projected_net_variance'] / 1e3, 1),
        'Proj. Net Var (%)':     round(s['projected_net_variance_pct'], 2),
        'Items At Risk':         s['n_at_risk'],
    }

pd.DataFrame(scenarios).T

**Observation:** At-risk count doesn't change with budget growth in this dataset. That's expected — the risk flag depends on cumulative variance as a *percentage* of combined two-quarter budget. When the projected budget grows, both the numerator (dollar run-rate variance) and denominator (budget) are unchanged — the cumulative percentage is stable. Only if the next-quarter budget changed *differentially* by line item would the risk profile shift.

### 6.3 The risk flag — cumulative two-quarter exposure

In [ ]:
from projection import RISK_THRESHOLD_PCT

print(f'Risk threshold: |Cumulative Var%| >= {RISK_THRESHOLD_PCT*100:.0f}%')
print()

# Show At Risk items and why they're flagged
at_risk = proj_df[proj_df['At Risk']]
print(f'Items At Risk: {len(at_risk)}')
print()

for _, row in at_risk.iterrows():
    two_q_budget = row['Budget'] + row['Next Q Budget']
    cumulative_var = row['Cumulative Variance ($)']
    cum_pct = row['Cumulative Variance (%)']
    print(f"  {row['Line Item']}")
    print(f"    Current Var ($):     ${row['Variance ($)']:>10,.0f}")
    print(f"    Projected Var ($):   ${row['Projected Variance ($)']:>10,.0f}")
    print(f"    Cumulative Var ($):  ${cumulative_var:>10,.0f}")
    print(f"    Two-quarter budget:  ${two_q_budget:>10,.0f}")
    print(f"    Cumulative Var (%):  {cum_pct:>+10.1f}%  (threshold: >{RISK_THRESHOLD_PCT*100:.0f}%)")
    print(f"    Unfavorable:         {not row['Favorable']}")
    print()

**Why cumulative two-quarter exposure?**

A single-quarter variance might be a one-time event — seasonality, a delayed contract, a vendor invoice timing issue. The cumulative view says: "If this pattern repeats, by the end of next quarter you'll have missed your target by X% *over two quarters*." That's a structurally different signal from a one-quarter miss.

An item is flagged At Risk only when **both** conditions are true:
1. The variance is **unfavorable** (favorable overperformance shouldn't be a "risk")
2. The **cumulative** two-quarter percentage exceeds the threshold

Condition 2 alone would flag an item that already overperformed in Q1 but is slightly under in Q2 — the cumulative might look bad in percentage terms even though the net position is fine. The unfavorable flag ensures we're only surfacing genuine negative risks.

In [ ]:
# Render the projection chart
from charts import plot_projection

fig = plot_projection(
    proj_df,
    title=f'Q4 2024 → {next_q}  ·  Variance Run-Rate Projection'
)
fig.show()

**Reading the chart:**
- **Solid bars** = current quarter variance (what actually happened)
- **Hatched bars** = projected next-quarter variance (run-rate assumption)
- Items prefixed with **⚠️** are flagged At Risk (unfavorable and cumulative breach ≥ 20%)
- Non-material items show no projected bar (they normalize to zero)

### 6.4 Next-quarter label inference

In [ ]:
# The next_quarter_label function handles common period formats
test_cases = [
    ('Q4 2024', 'Q1 2025'),  # Year rollover
    ('Q3 2024', 'Q4 2024'),  # Mid-year
    ('Q1 2025', 'Q2 2025'),  # Normal quarter
    ('H1 2024', 'H2 2024'),  # Half-year
    ('H2 2024', 'H1 2025'),  # Half-year rollover
    ('FY2024',  'FY2025'),   # Full year
    ('2024 Q2', 'Next Quarter'),  # Unrecognised format → fallback
]

print(f'{"Input":<15} {"Expected":<15} {"Got":<15} {"Match"}')
print('-' * 60)
for period, expected in test_cases:
    result = next_quarter_label(period)
    match = '✓' if result == expected else '✗'
    print(f'{period:<15} {expected:<15} {result:<15} {match}')

---

## 7. Conclusion

### What this approach enables vs. traditional FP&A reporting

| Traditional spreadsheet | This tool |
|------------------------|----------|
| Analyst spends a day formatting | Upload CSV → analysis in < 5 seconds |
| All 20+ rows shown equally | Materiality filter surfaces the 6-8 rows that matter |
| Variance column with no judgment | Favorability flag applies correct sign convention per category |
| Numbers only — no narrative | Three tone-controlled commentary paths from the same data |
| Backward-looking only | Forward projection with cumulative risk flags |
| Static deck, rebuilt each quarter | Parameterised: change the period, thresholds, or growth assumption and everything regenerates |
| One presentation format | Board deck, team brief, investor talking points — generated simultaneously |

### Design principles that guided the implementation

**1. Every number on screen should be explainable in one sentence to a CFO.**

This was the filter for every algorithmic decision. Dollar run-rate projection passes ("we're assuming the same dollar miss repeats"). A statistical ARIMA forecast would fail — no CFO should have to understand ARIMA to trust a number in a board deck.

**2. Computation separated from presentation.**

`variance_engine.py`, `commentary.py`, and `projection.py` contain zero Streamlit code. `charts.py` contains zero business logic. `app.py` is orchestration only. This makes every module independently testable and means a future API endpoint, CLI tool, or different frontend requires no changes to the analytical core.

**3. Graceful degradation over hard dependencies.**

- No API key? Rule-based commentary still runs.
- No kaleido? Charts render in the UI; PPTX shows a placeholder text instead of crashing.
- Budget = 0? Variance (%) returns NaN and severity defaults to Minor instead of dividing by zero.

**4. The dual-threshold materiality model is not intuitive but it's correct.**

Every FP&A team we surveyed used a single percentage threshold. The dual threshold (percentage OR absolute dollar) was the insight that prevented both false negatives (large absolute misses below the % threshold) and false positives (tiny budget lines that are noisy in percentage terms). The slider in the UI makes both thresholds configurable so teams can tune to their own definition of materiality.

### What this doesn't do (and why)

- **No statistical forecasting.** ARIMA, Prophet, and similar models require historical time series. A single-period P&L upload doesn't have that. The run-rate projection is deliberately simple — transparent math that doesn't require a PhD to audit.

- **No GL code or cost center mapping.** That requires an ERP integration, which is a separate layer. This tool works on the export that any ERP system can produce.

- **No multi-period trending.** A single-period variance file is the input contract. Multi-period analysis would require a different schema and a different set of questions (trend vs. point-in-time). That's a v2 scope.

- **No automated root cause detection.** The tool provides root-cause *hypotheses* (clearly labelled as such in the AI prompts), not root-cause *determination*. The business context required for genuine root cause attribution — headcount changes, pricing decisions, vendor contract renegotiations — is outside the data. Pretending otherwise would undermine CFO trust.

In [ ]:
# Final summary — the whole pipeline in one cell
from variance_engine import validate_csv, compute_variances, summarize_by_category, build_waterfall_data
from projection import project_next_quarter, summarize_projection, next_quarter_label

df_raw   = pd.read_csv(csv_path)
_ok, _   = validate_csv(df_raw)
enriched = compute_variances(df_raw)
cat_df   = summarize_by_category(enriched)
wf_df    = build_waterfall_data(enriched)
proj_df  = project_next_quarter(enriched, next_q_growth_pct=0.0)
summary  = summarize_projection(proj_df, 'Q4 2024', next_quarter_label('Q4 2024'))

print('Pipeline summary — Q4 2024')
print('=' * 50)
print(f'  Total Budget  : ${enriched["Budget"].sum():>15,.0f}')
print(f'  Total Actual  : ${enriched["Actual"].sum():>15,.0f}')
print(f'  Net Variance  : ${enriched["Actual"].sum() - enriched["Budget"].sum():>+15,.0f}')
print()
print(f'  Line items     : {len(enriched)}')
print(f'  Material items : {enriched["Material"].sum()}')
print(f'  Critical sev.  : {(enriched["Severity"] == "Critical").sum()}')
print(f'  Unfav. material: {(enriched["Material"] & ~enriched["Favorable"]).sum()}')
print()
print(f'  Waterfall bars : {len(wf_df)} (incl. Budget/Actual anchors)')
print()
print(f'  Projection to  : {next_quarter_label("Q4 2024")}')
print(f'  Proj. variance : ${summary["projected_net_variance"]:>+15,.0f}')
print(f'  Items at risk  : {summary["n_at_risk"]}')
if summary['at_risk_items']:
    for item in summary['at_risk_items']:
        print(f'    • {item}')